# Phase 5 — Grade Mapping & Threshold Calibration
Map predicted_day → A/B/C/D และ calibrate boundary ให้ accuracy สูงสุด

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.model import prepare_features, train_cv, FEATURE_COLS
from src.grade import day_to_grade, true_day_to_grade, evaluate_grades, calibrate_thresholds, THRESHOLDS, GRADE_ORDER

df = prepare_features('../data/features.csv')
print('Grade boundaries (default):')
for g, (lo, hi) in THRESHOLDS.items():
    days = [d for d in range(9) if lo <= d < hi]
    print(f'  {g}: day {lo:.1f}-{hi:.1f}  ({days})')

## Train model & get OOF predictions

In [ ]:
print('Training...')
model, cv = train_cv(df, n_splits=5)
oof_pred = cv['oof_pred']
print(f"\nCV MAE={cv['mae_mean']:.3f}  R2={cv['r2_mean']:.3f}")

## Default threshold — Accuracy & Confusion Matrix

In [ ]:
result = evaluate_grades(df, oof_pred)
print(f"Overall accuracy: {result['overall_accuracy']:.1%}")
print(f"COS accuracy:     {result['per_variety']['COS']:.1%}")
print(f"GOK accuracy:     {result['per_variety']['GOK']:.1%}")
print()
cm = result['confusion_matrix']
print(cm)

fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('Confusion Matrix — Default Threshold (OOF)')
plt.tight_layout()
plt.savefig('../results/grade_confusion_default.png', dpi=80, bbox_inches='tight')
plt.show()

## Calibrate thresholds (Grid Search)

In [ ]:
print('Calibrating...')
best = calibrate_thresholds(df, oof_pred, step=0.1)
print(f"\nBest thresholds:")
print(f"  A/B: {best['best_t1']}")
print(f"  B/C: {best['best_t2']}")
print(f"  C/D: {best['best_t3']}")
print(f"  Accuracy: {best['best_accuracy']:.1%}")

## Confusion Matrix หลัง Calibrate

In [ ]:
from src.grade import _apply_thresholds

cal_pred_grade = _apply_thresholds(oof_pred, best['best_t1'], best['best_t2'], best['best_t3'])
true_grade = true_day_to_grade(df['day'].values)

cm_cal = pd.crosstab(
    pd.Series(true_grade, name='True'),
    pd.Series(cal_pred_grade, name='Predicted'),
).reindex(index=GRADE_ORDER, columns=GRADE_ORDER, fill_value=0)

print(f'Calibrated accuracy: {np.mean(cal_pred_grade == true_grade):.1%}')
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (cm_plot, title) in zip(axes, [
    (result['confusion_matrix'], 'Default'),
    (cm_cal, 'Calibrated')
]):
    sns.heatmap(cm_plot, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5)
    ax.set_title(f'{title} Threshold')
plt.tight_layout()
plt.savefig('../results/grade_confusion_compare.png', dpi=80, bbox_inches='tight')
plt.show()

## Accuracy per Grade

In [ ]:
for grade in GRADE_ORDER:
    mask = true_grade == grade
    acc = np.mean(cal_pred_grade[mask] == true_grade[mask])
    n = mask.sum()
    print(f'Grade {grade}: {acc:.1%}  (n={n})')

## สรุป Phase 5

In [ ]:
grade_notes = {
    'grade_def': {'A':'D0-D1 สดเลย','B':'D2-D3 เหลืองนิดเดียว','C':'D4-D5 เริ่มเหลือง','D':'D6-D8 เหลือง/กินไม่ได้'},
    'default_accuracy': round(result['overall_accuracy'], 4),
    'calibrated_accuracy': best['best_accuracy'],
    'final_thresholds': {'A/B': best['best_t1'], 'B/C': best['best_t2'], 'C/D': best['best_t3']},
    'separate_variety_threshold': False,
    'notes': '',  # กรอกหลังดู confusion matrix
}
for k, v in grade_notes.items():
    print(f'{k}: {v}')